# Check binary classification results

`FOLDER_NAMES`에 `Demo_binary/checkpoints` 아래의 폴더명을 입력하면 각 실행의 시간 범위와 test accuracy 평균 ± 표준편차를 출력합니다.

In [53]:
from pathlib import Path
import json
from statistics import mean, stdev

CHECKPOINT_ROOT = Path("checkpoints_eegnet")
CHECKPOINT_ROOT_EMG = Path("checkpoints_emg_eegnet")


# 경로를 입력받고, 그 안에 있는 폴더를 리스트로 반환하는 함수
def get_checkpoint_folders(root: Path):
    return sorted([folder.name for folder in root.iterdir() if folder.is_dir()])


In [54]:

# 확인할 체크포인트 폴더명을 입력하세요.
FOLDER_NAMES = get_checkpoint_folders(CHECKPOINT_ROOT)
FOLDER_NAMES_EMG = get_checkpoint_folders(CHECKPOINT_ROOT_EMG)

print(FOLDER_NAMES)

['20260609_105630', '20260609_113615', '20260609_121357', '20260609_124230', '20260609_130527', '20260609_133448', '20260609_140335']


In [55]:
def print_checkpoint_results(folder_names, checkpoint_root):
    if isinstance(folder_names, str):
        folder_names = [folder_names]

    for folder_name in folder_names:
        folder = checkpoint_root / folder_name
        config_path = folder / "run_config.json"
        results_path = folder / "results.json"

        if not config_path.is_file():
            raise FileNotFoundError(
                f"{config_path}: run_config.json이 없습니다."
            )
        if not results_path.is_file():
            raise FileNotFoundError(
                f"{results_path}: results.json이 없습니다."
            )

        with config_path.open(encoding="utf-8") as file:
            time_range_s = json.load(file)["time_range_s"]

        with results_path.open(encoding="utf-8") as file:
            results = json.load(file)

        test_accs = [result["test_acc"] for result in results.values()]
        if not test_accs:
            raise ValueError(f"{folder_name}: test_acc 값이 없습니다.")

        acc_mean = mean(test_accs)
        acc_std = stdev(test_accs) if len(test_accs) > 1 else 0.0

        # print(f"Folder       : {folder_name}")
        print(f"time_range_s : {time_range_s}")
        print(f"test_acc     : {acc_mean:.4f} ± {acc_std:.4f} (n={len(test_accs)})")
        print()

print("######## Results for EEG checkpoints ########", '\n')
print_checkpoint_results(FOLDER_NAMES, CHECKPOINT_ROOT)
print("\n\n")
print("######## Results for EMG checkpoints ########", '\n')
print_checkpoint_results(FOLDER_NAMES_EMG, CHECKPOINT_ROOT_EMG)

######## Results for EEG checkpoints ######## 

time_range_s : [2.25, 2.75]
test_acc     : 0.5555 ± 0.0426 (n=10)

time_range_s : [2.5, 3.0]
test_acc     : 0.5651 ± 0.0256 (n=10)

time_range_s : [2.75, 3.25]
test_acc     : 0.5294 ± 0.0710 (n=10)

time_range_s : [3.0, 3.5]
test_acc     : 0.5094 ± 0.0492 (n=10)

time_range_s : [3.25, 3.75]
test_acc     : 0.5028 ± 0.0477 (n=10)

time_range_s : [3.5, 4.0]
test_acc     : 0.5063 ± 0.0586 (n=10)

time_range_s : [3.75, 4.25]
test_acc     : 0.5378 ± 0.0336 (n=10)




######## Results for EMG checkpoints ######## 

time_range_s : [2.25, 2.75]
test_acc     : 0.5458 ± 0.0607 (n=10)

time_range_s : [2.5, 3.0]
test_acc     : 0.5354 ± 0.0516 (n=10)

time_range_s : [2.75, 3.25]
test_acc     : 0.5242 ± 0.0434 (n=10)

time_range_s : [3.0, 3.5]
test_acc     : 0.5116 ± 0.0392 (n=10)

time_range_s : [3.25, 3.75]
test_acc     : 0.4937 ± 0.0430 (n=10)

time_range_s : [3.5, 4.0]
test_acc     : 0.5230 ± 0.0358 (n=10)

time_range_s : [3.75, 4.25]
test_acc     :

## Label / label-pair distinguishability

- 한 label의 기대 정확도: 해당 label이 포함된 모든 pair 모델의 `test_acc` 평균
- 두 label 사이의 기대 정확도: 해당 pair 모델의 `test_acc`
- 기본적으로 10개 모델을 모두 사용합니다. 제외할 모델이 명확할 때만 `EXCLUDED_MODELS`에 pair 이름을 입력하세요.

> `results.json`에는 샘플별 예측값이 없으므로 실제 10-model ensemble 정확도는 계산할 수 없습니다.

In [56]:
# 예: {"01", "24"}. 기본값은 제외 모델 없음.
EXCLUDED_MODELS = set()


def print_distinguishability(folder_names, excluded_models=None):
    if isinstance(folder_names, str):
        folder_names = [folder_names]
    excluded_models = set(excluded_models or [])

    for folder_name in folder_names:
        folder = CHECKPOINT_ROOT / folder_name
        with (folder / "run_config.json").open(encoding="utf-8") as file:
            time_range_s = json.load(file)["time_range_s"]
        with (folder / "results.json").open(encoding="utf-8") as file:
            results = json.load(file)

        pair_accs = {
            pair: result["test_acc"]
            for pair, result in results.items()
            if pair not in excluded_models
        }
        if not pair_accs:
            raise ValueError(f"{folder_name}: 사용할 모델이 없습니다.")

        labels = sorted({label for pair in pair_accs for label in pair})
        label_scores = {}
        for label in labels:
            relevant_accs = [acc for pair, acc in pair_accs.items() if label in pair]
            label_scores[label] = mean(relevant_accs)

        sorted_labels = sorted(label_scores.items(), key=lambda item: item[1], reverse=True)
        sorted_pairs = sorted(pair_accs.items(), key=lambda item: item[1], reverse=True)

        print(f"Folder: {folder_name} | time_range_s: {time_range_s}")
        print(f"Used models: {len(pair_accs)} | Excluded: {sorted(excluded_models)}")
        print("\nExpected accuracy by label")
        for label, score in sorted_labels:
            used_pairs = [pair for pair in pair_accs if label in pair]
            print(f"  label {label}: {score:.4f}  (pairs: {', '.join(used_pairs)})")
        print(f"Best label: {sorted_labels[0][0]} ({sorted_labels[0][1]:.4f})")

        print("\nExpected accuracy by label pair")
        for pair, score in sorted_pairs:
            print(f"  label {pair[0]} vs {pair[1]}: {score:.4f}")
        print(f"Best pair: label {sorted_pairs[0][0][0]} vs {sorted_pairs[0][0][1]} "
              f"({sorted_pairs[0][1]:.4f})")
        print("\n" + "-" * 60 + "\n")


print_distinguishability(FOLDER_NAMES, EXCLUDED_MODELS)

Folder: 20260609_105630 | time_range_s: [2.25, 2.75]
Used models: 10 | Excluded: []

Expected accuracy by label
  label 2: 0.5667  (pairs: 02, 12, 23, 24)
  label 1: 0.5613  (pairs: 01, 12, 13, 14)
  label 4: 0.5586  (pairs: 04, 14, 24, 34)
  label 0: 0.5480  (pairs: 01, 02, 03, 04)
  label 3: 0.5427  (pairs: 03, 13, 23, 34)
Best label: 2 (0.5667)

Expected accuracy by label pair
  label 1 vs 2: 0.6208
  label 0 vs 4: 0.5935
  label 3 vs 4: 0.5887
  label 0 vs 1: 0.5713
  label 1 vs 3: 0.5569
  label 2 vs 4: 0.5562
  label 0 vs 2: 0.5458
  label 2 vs 3: 0.5440
  label 1 vs 4: 0.4961
  label 0 vs 3: 0.4813
Best pair: label 1 vs 2 (0.6208)

------------------------------------------------------------

Folder: 20260609_113615 | time_range_s: [2.5, 3.0]
Used models: 10 | Excluded: []

Expected accuracy by label
  label 4: 0.5829  (pairs: 04, 14, 24, 34)
  label 2: 0.5663  (pairs: 02, 12, 23, 24)
  label 3: 0.5659  (pairs: 03, 13, 23, 34)
  label 0: 0.5587  (pairs: 01, 02, 03, 04)
  label 1

## 5-class ensemble

각 binary 모델의 log-probability를 결합해 5-class label을 예측합니다. Validation label accuracy의 macro 평균이 가장 높은 모델 조합을 선택하고, 선택에 사용하지 않은 test set에서 label별 accuracy를 평가합니다.

In [58]:
import sys
from itertools import combinations
import numpy as np
import torch

TARGET_TIME_RANGE = [2.5, 3.0] 
INPUT_TYPE = "emg"  # "eeg" or "emg"
BATCH_SIZE = 32
MODEL_NAMES = ["01", "02", "03", "04", "12", "13", "14", "23", "24", "34"]

DEMO_BINARY_DIR = Path("Demo_binary") if Path("Demo_binary").is_dir() else Path(".")
sys.path.insert(0, str(DEMO_BINARY_DIR.resolve()))
from train_10_binary_eegnet import EEGNet

checkpoint_root = CHECKPOINT_ROOT_EMG if INPUT_TYPE == "emg" else CHECKPOINT_ROOT
folder_names = FOLDER_NAMES_EMG if INPUT_TYPE == "emg" else FOLDER_NAMES

# TARGET_TIME_RANGE 체크포인트 찾기
run_folder = None
for folder_name in folder_names:
    folder = checkpoint_root / folder_name
    with (folder / "run_config.json").open(encoding="utf-8") as file:
        config = json.load(file)
    if config["time_range_s"] == TARGET_TIME_RANGE:
        run_folder = folder
        break
if run_folder is None:
    raise ValueError(f"time_range_s={TARGET_TIME_RANGE}인 폴더가 {checkpoint_root}에 없습니다.")

data_dir = Path(config["args"]["data_dir"])
input_file = config["args"].get("input_file", "X_eeg_raw.npy")
X = np.load(data_dir / input_file, mmap_mode="r")  # shape: (N, C, T)
y = np.load(data_dir / "y.npy").astype(np.int64)
sfreq = float(config["sfreq"])
with (data_dir / "preprocess_meta.json").open(encoding="utf-8") as file:
    data_start_s = float(json.load(file)["epoch"]["crop_tmin"])
start_idx = int(round((TARGET_TIME_RANGE[0] - data_start_s) * sfreq))
end_idx = int(round((TARGET_TIME_RANGE[1] - data_start_s) * sfreq)) + 1
X = X[:, :, start_idx:end_idx]

train_size, val_size, test_size = config["split_sizes"]
val_idx = np.arange(train_size, train_size + val_size)
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)
train_mean = X[:train_size].mean(axis=(0, 2))[:, None].astype(np.float32)
train_std = np.maximum(X[:train_size].std(axis=(0, 2))[:, None], 1e-6).astype(np.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 각 모델의 validation/test log-probability 계산
log_probs = {"val": {}, "test": {}}
for model_name in MODEL_NAMES:
    model = EEGNet(X.shape[1]).to(device)
    state = torch.load(run_folder / f"y_{model_name}/best_model.pt", map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.eval()

    for split_name, indices in (("val", val_idx), ("test", test_idx)):
        outputs = []
        with torch.no_grad():
            for start in range(0, len(indices), BATCH_SIZE):
                batch_idx = indices[start:start + BATCH_SIZE]
                batch = ((X[batch_idx] - train_mean) / train_std).astype(np.float32)
                logits = model(torch.from_numpy(batch).to(device))
                outputs.append(torch.log_softmax(logits, dim=1).cpu().numpy())
        log_probs[split_name][model_name] = np.concatenate(outputs)


def ensemble_metrics(model_subset, split_name, indices):
    scores = np.zeros((len(indices), 5), dtype=np.float64)
    for model_name in model_subset:
        # 모델 출력 0: label이 model_name의 두 label 중 하나, 출력 1: 나머지 label
        expected_binary = np.array([0 if str(label) in model_name else 1 for label in range(5)])
        scores += log_probs[split_name][model_name][:, expected_binary]
    predictions = scores.argmax(axis=1)
    label_accs = np.array([(predictions[y[indices] == label] == label).mean() for label in range(5)])
    return label_accs, label_accs.mean(), (predictions == y[indices]).mean()


# 모든 label을 고유하게 구분할 수 있는 모델 부분집합을 전수 검사
candidates = []
for size in range(1, len(MODEL_NAMES) + 1):
    for subset in combinations(MODEL_NAMES, size):
        label_codes = {
            tuple(0 if str(label) in model_name else 1 for model_name in subset)
            for label in range(5)
        }
        if len(label_codes) < 5:
            continue
        val_label_accs, val_macro_acc, val_acc = ensemble_metrics(subset, "val", val_idx)
        test_label_accs, test_macro_acc, test_acc = ensemble_metrics(subset, "test", test_idx)
        candidates.append({
            "subset": subset,
            "val_label_accs": val_label_accs,
            "val_macro": val_macro_acc,
            "val_acc": val_acc,
            "test_label_accs": test_label_accs,
            "test_macro": test_macro_acc,
            "test_acc": test_acc,
        })

best_by_val = max(candidates, key=lambda x: (x["val_macro"], x["val_acc"], -len(x["subset"])))
best_by_test = max(candidates, key=lambda x: (x["test_macro"], x["test_acc"], -len(x["subset"])))
all_val_label_accs, all_val_macro, all_val_acc = ensemble_metrics(MODEL_NAMES, "val", val_idx)
all_test_label_accs, all_test_macro, all_test_acc = ensemble_metrics(MODEL_NAMES, "test", test_idx)

print(f"Input: {input_file} | folder: {run_folder.name} | time_range_s: {TARGET_TIME_RANGE} | device: {device}")
print("\nAll 10 models")
print(f"  validation macro/overall: {all_val_macro:.4f} / {all_val_acc:.4f}")
print(f"  test label accs: " + ", ".join(f"label {i}={acc:.4f}" for i, acc in enumerate(all_test_label_accs)))
print(f"  test macro/overall: {all_test_macro:.4f} / {all_test_acc:.4f}")

print("\nBest combination selected by validation accuracy")
print(f"  models: {list(best_by_val['subset'])}")
print(f"  validation label accs: " + ", ".join(f"label {i}={acc:.4f}" for i, acc in enumerate(best_by_val["val_label_accs"])))
print(f"  validation macro/overall: {best_by_val['val_macro']:.4f} / {best_by_val['val_acc']:.4f}")
print(f"  test label accs: " + ", ".join(f"label {i}={acc:.4f}" for i, acc in enumerate(best_by_val["test_label_accs"])))
print(f"  test macro/overall: {best_by_val['test_macro']:.4f} / {best_by_val['test_acc']:.4f}")

Input: X_emg_raw.npy | folder: 20260609_194225 | time_range_s: [2.5, 3.0] | device: cuda

All 10 models
  validation macro/overall: 0.3189 / 0.3300
  test label accs: label 0=0.0000, label 1=0.1923, label 2=0.2500, label 3=0.0588, label 4=0.7059
  test macro/overall: 0.2414 / 0.2300

Best combination selected by validation accuracy
  models: ['01', '02', '03', '04', '12', '34']
  validation label accs: label 0=0.1600, label 1=0.4000, label 2=0.7727, label 3=0.4118, label 4=0.4286
  validation macro/overall: 0.4346 / 0.4300
  test label accs: label 0=0.0000, label 1=0.3077, label 2=0.4500, label 3=0.2353, label 4=0.2353
  test macro/overall: 0.2457 / 0.2500
